# Q-Former

Q-Former 把 ViT 产生的大量视觉特征，压缩成少量适合交给 LLM 的视觉 Token

### 模型结构

``` text
输入图像
   │
   ▼
┌─────────────┐
│  ViT / 视觉  │  → vision_features [B, N, vision_dim]
│  编码器      │     (N = 197 个 patch token)
└─────────────┘
   │
   ▼  Linear(vision_dim → query_dim)
   │
   ▼
┌─────────────────────────────────────────────┐
│              Q-Former (Cross-Attention)     │
│                                             │
│  Query Tokens [B, 32, D] ──┐                │
│  (可学习，32个"问题")       │                │
│                            ├─ Cross-Attn  ──┤
│  图像特征 [B, N, D] ────────┘                │
│  (N个patch，作为K/V)                         │
│                                             │
│  → 残差 + LayerNorm                         │
│  → 输出 [B, 32, D]                          │
└─────────────────────────────────────────────┘
   │
   ▼
┌─────────────┐
│   大语言模型  │  → 文本生成
│   (LLM)     │
└─────────────┘
```

### 模块解释

- projector：对齐 ViT 产生的视觉特征，变成 LLM 能理解的 token
- cross-attention：压缩 ViT 的 Token 数量到32个


In [ ]:
import torch
import torch.nn as nn


class SimpleQFormer(nn.Module):
    """
    轻量级 Q-Former（Querying Transformer）

    架构流程：
      Query Tokens (learnable) ─┐
                                 ├─→ Cross-Attention ─→ 残差 + LayerNorm ─→ 输出
      图像特征 (vision_features) ─┘

    Args:
        vision_dim:  图像编码器输出的特征维度（如 ViT 的 hidden_size）
        query_dim:   Query Token 和内部表示的维度
        num_queries: Query Token 的数量（默认 32，控制输出序列长度）
        num_heads:   交叉注意力的头数（默认 8）
    """
    def __init__(
        self,
        vision_dim: int,
        query_dim: int,
        num_queries: int = 32,
        num_heads: int = 8,
    ):
        super().__init__()

        self.query_tokens = nn.Parameter(
            torch.randn(1, num_queries, query_dim)
        )

        # 视觉特征投影层：将图像特征从 vision_dim 映射到 query_dim
        # 确保 Query 和图像特征在同一个维度空间，才能做交叉注意力
        self.vision_projection = nn.Linear(
            vision_dim,
            query_dim,
        )

        # 交叉注意力层：Query 作为 Q，图像特征作为 K 和 V
        # 每个 Query Token 学会从图像中"问"出自己关心的信息
        # batch_first=True：输入/输出形状为 [B, N, D] 而非 [N, B, D]
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=query_dim,
            num_heads=num_heads,
            batch_first=True,
        )

        self.norm = nn.LayerNorm(query_dim)

    def forward(
        self,
        vision_features: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            vision_features: [B, N, vision_dim]
                B = batch_size
                N = 图像 patch 数量（如 ViT 的 197 个 token）
                vision_dim = 每个 patch 的特征维度

        Returns:
            query_output: [B, num_queries, query_dim]
                压缩后的视觉特征，可直接作为 LLM 的输入前缀
        """
        batch_size = vision_features.shape[0]

        # ----- 1. 扩展 Query Token 到当前 batch -----
        # [1, num_queries, D] → [B, num_queries, D]
        # -1 表示保持该维度不变
        queries = self.query_tokens.expand(
            batch_size,
            -1,
            -1,
        )

        # ----- 2. 图像特征投影到 query_dim -----
        # [B, N, vision_dim] → [B, N, query_dim]
        vision_features = self.vision_projection(
            vision_features
        )

        # ----- 3. 交叉注意力：Query 提问，图像特征回答 -----
        # query=queries:       [B, num_queries, D]  ← 可学习的"问题"
        # key=vision_features:  [B, N,         D]  ← 图像 patch 特征
        # value=vision_features:[B, N,         D]  ← 图像 patch 特征
        #
        # 每个 Query Token 会对所有 N 个图像 patch 计算注意力权重，
        # 然后加权聚合，得到自己关心的信息
        query_output, attention_weights = (
            self.cross_attention(
                query=queries,
                key=vision_features,
                value=vision_features,
            )
        )

        # ----- 4. 残差连接 + LayerNorm -----
        # queries + query_output：保留原始 query 信息 + 注意力提取的视觉信息
        query_output = self.norm(
            queries + query_output
        )

        return query_output